# Phase 3: Local assistant with Pydantic AI and an MCP time server

This notebook connects Pydantic AI to the local Ollama OpenAI-compatible endpoint and starts the MCP time server as a local subprocess. The model can answer ordinary questions and use the time server for current-time requests.

Run cells from top to bottom. The Ollama model and Python packages must already be available in this environment.

In [ ]:
import sys

from pydantic_ai import Agent
from pydantic_ai.messages import ToolReturnPart
from pydantic_ai.mcp import MCPToolset
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from fastmcp.client.transports import StdioTransport

provider = OpenAIProvider(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

agent_model = OpenAIChatModel(
    model_name="qwen3.5:2b",
    provider=provider,
)

print("Local Ollama model configured.")

In [ ]:
time_server = MCPToolset(
    StdioTransport(
        command=sys.executable,
        args=[
            "-m",
            "mcp_server_time",
            "--local-timezone=Asia/Kolkata",
        ],
    )
)

agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt=(
        "You are a helpful assistant. For current date or time questions, always "
        "use the MCP time server. Infer an appropriate IANA time zone from the "
        "city, state, or country the user names, then call get_current_time "
        "directly with that time zone. Never use convert_time to work out the "
        "current time from another location. Never calculate, estimate, or "
        "invent a timestamp. For non-time questions, answer normally."
    ),
)

print("Agent and local MCP time server configured.")

## Run a question

For a time question, this function returns the actual result from the MCP get_current_time tool rather than relying on the model to rewrite the timestamp. If the model uses a conversion tool or skips the direct time tool, it reports that no direct timestamp was received.

In [ ]:
def content_to_text(content) -> str:
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            item.text if hasattr(item, "text") else str(item)
            for item in content
        )

    return str(content)


async def run_agent(prompt: str) -> str:
    result = await agent.run(prompt)
    asks_about_time = any(
        word in prompt.lower() for word in ("time", "date")
    )

    if asks_about_time:
        for message in result.new_messages():
            for part in message.parts:
                if (
                    isinstance(part, ToolReturnPart)
                    and "get_current_time" in part.tool_name.lower()
                ):
                    return content_to_text(part.content)

        return (
            "I couldn't get a direct current-time result. "
            "The time tool may not have been called."
        )

    return result.output

print("Agent function ready.")

## Try it

The first question tests ordinary local-model answering. The second tests time-zone inference and direct timestamp retrieval. The server returns the time to seconds; this notebook displays its tool result without having the model recalculate it.

In [ ]:
answer = await run_agent("What is the capital of American Samoa?")
print(answer)

answer = await run_agent("What is the current date and time in Ohio?")
print(answer)